[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/herrrickshaw/india-trade-data-analysis/blob/main/notebooks/repo_verification.ipynb)

# Repo Verification — India Trade Analysis (quick health check)

A fast, no-setup sanity check for both halves of the India trade analysis split:
- [`india-trade-data-analysis`](https://github.com/herrrickshaw/india-trade-data-analysis) — descriptive data analysis
- [`india-trade-sector-policy-recommendations`](https://github.com/herrrickshaw/india-trade-sector-policy-recommendations) — sector &amp; policy recommendations

Run all cells top to bottom (Runtime → Run all). It checks, live, right now:
1. Every chart/report/notebook file in both repos resolves via **GitHub Pages** (the served site, not just the raw git blob)
2. Every JSON dataset in both repos **parses** and has a non-trivial payload
3. The three **cross-repo links** inside `five_year_trade_currency_synthesis.html` (repo A → repo B charts) actually resolve
4. A one-line **PASS/FAIL summary** at the end — if anything's broken, you'll see exactly what and where

No local clone, no dependencies beyond `requests` (already in Colab).

## 1. Setup

In [ ]:
import requests

REPO_A = "india-trade-data-analysis"
REPO_B = "india-trade-sector-policy-recommendations"
PAGES_A = f"https://herrrickshaw.github.io/{REPO_A}"
PAGES_B = f"https://herrrickshaw.github.io/{REPO_B}"
RAW_A = f"https://raw.githubusercontent.com/herrrickshaw/{REPO_A}/main"
RAW_B = f"https://raw.githubusercontent.com/herrrickshaw/{REPO_B}/main"

results = []  # (repo, path, ok: bool, detail: str)


## 2. Check every file resolves via GitHub Pages

In [ ]:
FILES_A = [
    "index.html", "README.md",
    "charts/cpi_wpi_state_trends.html", "charts/cpi_india_heatmap.html",
    "charts/rbi_forex_reserves_trend.html", "charts/real_exchange_rate_and_currency_policy.html",
    "charts/trade_balance_hsn_analysis.html", "charts/hsn_historical_trends.html",
    "charts/fertiliser_fuel_price_transmission.html", "charts/gdp_growth_trend.html",
    "charts/five_year_trade_currency_synthesis.html",
    "reports/FY2025-26_sample_report.html", "reports/FY2025-26_sample_report.pdf",
    "notebooks/mospi_live_reference.ipynb",
]
FILES_B = [
    "index.html", "README.md",
    "charts/import_dependency_policy_gap_analysis.html", "charts/country_trade_deficit_and_policy_history.html",
    "charts/sector_country_priority_and_pli_coverage.html", "charts/export_destination_priority_and_pli_coverage.html",
    "charts/sector_and_policy_recommendations.html",
]

def check_files(repo_name, base_url, files):
    for f in files:
        try:
            r = requests.head(f"{base_url}/{f}", timeout=15, allow_redirects=True)
            ok = r.status_code == 200
            results.append((repo_name, f, ok, f"HTTP {r.status_code}"))
        except requests.RequestException as e:
            results.append((repo_name, f, False, str(e)))

check_files(REPO_A, PAGES_A, FILES_A)
check_files(REPO_B, PAGES_B, FILES_B)
print(f"Checked {len(FILES_A) + len(FILES_B)} files across both repos.")


## 3. Check every JSON dataset parses and isn't empty

In [ ]:
DATASETS_A = [
    "mospi_snapshot_2026-07-18", "cpi_statewise_trend_2025-01_to_2026-06", "wpi_national_trend_2025-01_to_2026-04",
    "rbi_forex_reserves_2015-01_to_2025-06", "rbi_usd_inr_exchange_rate_2015-01_to_2026-07",
    "rupee_real_exchange_rate_2015_to_2026", "tradestat_hsn_export_import_2018-19_to_2025-26",
    "hsn_historical_trends_2018-19_to_2025-26", "services_trade_and_overall_balance_2015-16_to_2024-25",
    "trade_data_cross_validation_2026-07-18", "fertiliser_fuel_price_transmission_2012_to_2026",
    "gdp_growth_rate_2012-13_to_2025-26", "five_year_trade_currency_synthesis_2026-07-18",
    "fy2025_26_sample_report_2026-07-18",
]
DATASETS_B = [
    "sector_country_priority_and_pli_coverage_2026-07-18", "export_destination_priority_and_pli_coverage_2026-07-18",
    "import_dependency_policy_gap_analysis_2026-07-18", "country_trade_deficit_and_policy_history_2026-07-18",
    "country_commodity_breakdown_2026-07-18", "sector_and_policy_recommendations_2026-07-18",
]

def check_json(repo_name, base_url, names):
    for name in names:
        path = f"data/{name}.json"
        try:
            r = requests.get(f"{base_url}/{path}", timeout=20)
            if r.status_code != 200:
                results.append((repo_name, path, False, f"HTTP {r.status_code}"))
                continue
            payload = r.json()
            ok = isinstance(payload, dict) and len(payload) > 0
            results.append((repo_name, path, ok, f"{len(payload)} top-level keys" if ok else "empty/malformed"))
        except Exception as e:
            results.append((repo_name, path, False, str(e)))

check_json(REPO_A, RAW_A, DATASETS_A)
check_json(REPO_B, RAW_B, DATASETS_B)
print(f"Checked {len(DATASETS_A) + len(DATASETS_B)} JSON datasets.")


## 4. Check the three cross-repo links (repo A's five-year synthesis → repo B's charts)

In [ ]:
CROSS_REPO_LINKS = [
    f"https://github.com/herrrickshaw/{REPO_B}/blob/main/charts/sector_country_priority_and_pli_coverage.html",
    f"https://github.com/herrrickshaw/{REPO_B}/blob/main/charts/export_destination_priority_and_pli_coverage.html",
    f"https://github.com/herrrickshaw/{REPO_B}/blob/main/charts/country_trade_deficit_and_policy_history.html",
]

# Confirm these three URLs are the ones actually embedded in repo A's five-year synthesis chart --
# not just that they resolve, but that they're really referenced there.
synth_html = requests.get(f"{RAW_A}/charts/five_year_trade_currency_synthesis.html", timeout=20).text
for url in CROSS_REPO_LINKS:
    referenced = url in synth_html
    try:
        r = requests.head(url, timeout=15, allow_redirects=True)
        resolves = r.status_code == 200
    except requests.RequestException:
        resolves = False
    ok = referenced and resolves
    results.append(("cross-repo link", url, ok, f"referenced={referenced}, resolves={resolves}"))
print("Checked 3 cross-repo links.")


## 5. Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["repo", "path", "ok", "detail"])
n_ok = df["ok"].sum()
n_total = len(df)

print(f"{'='*60}")
print(f"  {n_ok}/{n_total} checks passed")
print(f"{'='*60}")

failures = df[~df["ok"]]
if failures.empty:
    print("\nAll checks passed -- both repos are healthy.")
else:
    print(f"\n{len(failures)} FAILURE(S):")
    for _, row in failures.iterrows():
        print(f"  [{row['repo']}] {row['path']} -- {row['detail']}")

df
